In [1]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip install -q peft==0.18.1 bitsandbytes accelerate datasets pillow
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00


In [2]:
!pip install pandas

In [3]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
DATA_DIR = Path("/kaggle/input/competitions/pixels-to-predictions/images")
VAL_DATA_DIR = Path("/kaggle/input/competitions/pixels-to-predictions")
# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE = 224
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


In [4]:
# # ── 2a-continued. Weighted sampling setup ─────────────────────────────────────
# from collections import Counter
# from torch.utils.data import WeightedRandomSampler

# # Keep the full original train_df.
# # Do NOT overwrite train_df with a small stratified sample.
# print(f"Original train size: {len(train_df):,}")
# print(f"Val size: {len(val_df):,}")
# print(f"Test size: {len(test_df):,}")

# # Compute-limited training budget.
# # This controls how many samples the model sees per epoch.
# TRAIN_SAMPLES_PER_EPOCH = 1000   

# # Create strata using subject + grade.
# # This gives rare subject/grade combinations more probability,
# # but does not force perfect balancing.
# strata = (
#     train_df["subject"].astype(str)
#     + " | "
#     + train_df["grade"].astype(str)
# ).tolist()

# stratum_counts = Counter(strata)

# # Mild inverse weighting.
# # sqrt prevents tiny strata from being oversampled too aggressively.
# sample_weights = [
#     1.0 / (stratum_counts[s] ** 0.5)
#     for s in strata
# ]

# # Optional: make the sampler reproducible
# sampler_generator = torch.Generator()
# sampler_generator.manual_seed(SEED)

# weighted_train_sampler = WeightedRandomSampler(
#     weights=torch.DoubleTensor(sample_weights),
#     num_samples=TRAIN_SAMPLES_PER_EPOCH,
#     replacement=True,
#     generator=sampler_generator,
# )

# print(f"Weighted sampler will draw {TRAIN_SAMPLES_PER_EPOCH:,} samples per epoch.")
# print(f"Number of subject-grade strata: {len(stratum_counts)}")

# print("\nSmallest strata:")
# for stratum, count in sorted(stratum_counts.items(), key=lambda x: x[1])[:10]:
#     print(f"  {stratum:<35} {count}")

In [5]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv("/kaggle/input/competitions/pixels-to-predictions/train.csv")
val_df   = pd.read_csv("/kaggle/input/competitions/pixels-to-predictions/val.csv")
test_df  = pd.read_csv("/kaggle/input/competitions/pixels-to-predictions/test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)
    
TRAIN_SAMPLES_PER_EPOCH = 100  
#for baseline, only using a subset of the data:
# train_df = train_df.sample(n=500, random_state=SEED).reset_index(drop=True)
# val_df = train_df.sample(n=500, random_state=SEED).reset_index(drop=True)
# test_df = train_df.sample(n=500, random_state=SEED).reset_index(drop=True)

#sampling stuff:

In [6]:
# #just subject sampling
# from collections import Counter
# from torch.utils.data import WeightedRandomSampler
# import torch

# TRAIN_SAMPLES_PER_EPOCH = 800

# subjects = train_df["subject"].astype(str).tolist()
# subject_counts = Counter(subjects)

# sample_weights = [
#     1.0 / (subject_counts[s] ** 0.5)
#     for s in subjects
# ]

# sampler_generator = torch.Generator()
# sampler_generator.manual_seed(SEED)

# weighted_train_sampler = WeightedRandomSampler(
#     weights=torch.DoubleTensor(sample_weights),
#     num_samples=TRAIN_SAMPLES_PER_EPOCH,
#     replacement=True,
#     generator=sampler_generator,
# )

In [7]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Builds the text prompt for the Vision Language Model.
    The <image> token is required for the model to process the image.
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    #prompt += f"Subject: {row['subject']} | Grade: {row['grade']}\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

# Display an example prompt
print(build_prompt(train_df.iloc[0], include_answer=True))

<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't always successful

In [8]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img = Image.open(self.data_dir / rel_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }
train_subset_df = train_df.sample(n=TRAIN_SAMPLES_PER_EPOCH, random_state=SEED).reset_index(drop=True)

train_ds = ScienceQADataset(train_subset_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

Datasets created: train=100, val=1048, test=1008


In [9]:
# ── 3a. Load SmolVLM model + run one inference example ───────────────────────
#orig import :from transformers import AutoProcessor, AutoModelForVision2Seq
#from transformers import AutoProcessor,AutoModelForImageTextToText
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import prepare_model_for_kbit_training
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
# model = AutoModelForVision2Seq.from_pretrained(
#     MODEL_ID,
#     torch_dtype=dtype,
#     device_map="auto" if torch.cuda.is_available() else None,
#     low_cpu_mem_usage=True,
#  )
# model = AutoModelForImageTextToText.from_pretrained(
#     MODEL_ID,
#     torch_dtype=dtype,
#     device_map="auto" if torch.cuda.is_available() else None,
#     low_cpu_mem_usage=True,
# )

#quantization to solve OOM error:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model = prepare_model_for_kbit_training(model)
if not torch.cuda.is_available():
    model.to(device)
model.eval()

# Pick a sample from validation set
sample = val_df.iloc[0]
sample_image = Image.open(DATA_DIR / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(
    text=[sample_prompt],
    images=[sample_image],
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Prompt:")
print(sample_prompt)
print("\nModel output:")
print(decoded)
print(f"\nGround-truth answer index: {sample['answer']}")

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Prompt:
<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't always su

In [10]:
# ── 4. LoRA Finetuning Setup ─────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

# ── 4a. Apply LoRA ────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,#after expanding mlp layers, decreasing rank to 8 from 16
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],#target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
)

model.train()
model = get_peft_model(model, lora_config)

# Optional, helpful for memory
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

# ── 4b. Collate function ─────────────────────────────────────────────────────
def collate_fn(batch):
    texts = [b["text"] for b in batch]
    images = [b["image"] for b in batch]
    answers = [b.get("answer",-1) for b in batch]

    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    )

    labels = inputs["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    # Mask everything up to and including "Answer:"
    answer_token_ids = processor.tokenizer.encode("Answer:", add_special_tokens=False)

    for i, seq in enumerate(labels):
        seq_list = seq.tolist()
        for j in range(len(seq_list) - len(answer_token_ids), -1, -1):
            if seq_list[j : j + len(answer_token_ids)] == answer_token_ids:
                labels[i, : j + len(answer_token_ids)] = -100
                break

        
    inputs["labels"] = labels
    inputs["answers"] = answers
    return {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

# ── 4c. DataLoaders ───────────────────────────────────────────────────────────
BATCH_SIZE = 1
NUM_EPOCHS = 3
LR = 2e-4

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    #sampler=weighted_train_sampler,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False,
)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01,
)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS * len(train_loader),
)

# ── 4d. Evaluation helper ─────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(loader,max_batches=200):
    model.eval()
    correct, total = 0, 0
    first_device = next(model.parameters()).device

    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        gt_answers = batch.pop("answers")
        gt_letters = [chr(ord("A") + a) for a in gt_answers]
        

        batch.pop("labels", None)

        inputs = {
            k: v.to(first_device) if torch.is_tensor(v) else v
            for k, v in batch.items()
        }

        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
        )

        # Decode only newly generated tokens
        input_len = inputs["input_ids"].shape[1]
        generated_only = outputs[:, input_len:]

        decoded = processor.tokenizer.batch_decode(
            generated_only,
            skip_special_tokens=True,
        )

        for j, text in enumerate(decoded):
            text_upper = text.strip().upper()

            pred = ""
            for ch in text_upper:
                if ch in "ABCDE":
                    pred = ch
                    break

            correct += int(pred == gt_letters[j])
            total += 1

    model.train()
    return correct / total if total > 0 else 0.0

trainable params: 4,784,128 || all params: 512,266,432 || trainable%: 0.9339


In [11]:
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working")
CKPT_DIR = OUTPUT_DIR / "run11"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

best_val_acc = -1.0

for epoch in range(NUM_EPOCHS):
    model.train()

    for step, batch in enumerate(train_loader, start=1):
        batch = {
            k: v.to(model.device) if torch.is_tensor(v) else v
            for k, v in batch.items()
        }
        batch.pop("answers",None)
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        if step % 20 == 0:
            lr = scheduler.get_last_lr()[0]
            print(
                f"[Epoch {epoch+1}/{NUM_EPOCHS}] "
                f"step {step}/{len(train_loader)} "
                f"loss={loss.item():.4f}  lr={lr:.2e}"
            )

    # Run validation AFTER the epoch finishes
    val_acc = evaluate(val_loader)

    #val_acc = evaluate(val_loader, max_batches=200)

    print(f"\n=== Epoch {epoch+1} done | val_acc={val_acc:.4f} ===")

    # Save best LoRA checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(CKPT_DIR)
        processor.save_pretrained(CKPT_DIR)
        print(f"✓ New best saved to {CKPT_DIR}")

print(f"\nFinetuning complete. Best val accuracy: {best_val_acc:.4f}")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered intern

[Epoch 1/3] step 20/100 loss=2.5804  lr=1.98e-04
[Epoch 1/3] step 40/100 loss=1.0083  lr=1.91e-04
[Epoch 1/3] step 60/100 loss=5.4203  lr=1.81e-04
[Epoch 1/3] step 80/100 loss=1.0314  lr=1.67e-04
[Epoch 1/3] step 100/100 loss=0.0000  lr=1.50e-04

=== Epoch 1 done | val_acc=0.4650 ===
✓ New best saved to /kaggle/working/run11
[Epoch 2/3] step 20/100 loss=1.1350  lr=1.31e-04
[Epoch 2/3] step 40/100 loss=0.0000  lr=1.10e-04
[Epoch 2/3] step 60/100 loss=0.0091  lr=8.95e-05
[Epoch 2/3] step 80/100 loss=0.1073  lr=6.91e-05
[Epoch 2/3] step 100/100 loss=0.0000  lr=5.00e-05

=== Epoch 2 done | val_acc=0.5300 ===
✓ New best saved to /kaggle/working/run11
[Epoch 3/3] step 20/100 loss=0.0000  lr=3.31e-05
[Epoch 3/3] step 40/100 loss=0.0000  lr=1.91e-05
[Epoch 3/3] step 60/100 loss=0.0000  lr=8.65e-06
[Epoch 3/3] step 80/100 loss=0.0011  lr=2.19e-06
[Epoch 3/3] step 100/100 loss=0.0000  lr=0.00e+00

=== Epoch 3 done | val_acc=0.5200 ===

Finetuning complete. Best val accuracy: 0.5300


In [12]:
from peft import PeftModel
import re
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForImageTextToText

# Load best LoRA checkpoint
BASE_MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
LORA_DIR = "/kaggle/working/run11"

processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()

sample_submission = pd.read_csv(
    "/kaggle/input/competitions/pixels-to-predictions/sample_submission.csv"
)

# Reorder test_df to match sample_submission exactly
test_for_submission = sample_submission[["id"]].merge(
    test_df,
    on="id",
    how="left"
)

print(test_for_submission.shape)
print(test_for_submission[["id"]].head())

def extract_answer_letter(text):
    """
    Extracts A/B/C/... from generated text.
    """
    if "Answer:" in text:
        text = text.split("Answer:")[-1].strip()

    match = re.search(r"\b[A-J]\b", text.upper())
    if match:
        return match.group(0)

    return "A"  # fallback

def predict_one(row):
    image = Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    prompt = build_prompt(row, include_answer=False)

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )

    first_device = next(model.parameters()).device
    inputs = {
        k: v.to(first_device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
        )

    decoded = processor.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    letter = extract_answer_letter(decoded)

    return CHOICE_LETTERS.index(letter)

preds = []

for _, row in tqdm(test_for_submission.iterrows(), total=len(test_for_submission)):
    preds.append(predict_one(row))

submission = pd.DataFrame({
    "id": sample_submission["id"],
    "answer": preds
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.shape)
print(submission.head())

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

(1008, 13)
           id
0  test_01750
1  test_00128
2  test_02891
3  test_02425
4  test_00930


  0%|          | 0/1008 [00:00<?, ?it/s]

(1008, 2)
           id  answer
0  test_01750       2
1  test_00128       1
2  test_02891       1
3  test_02425       1
4  test_00930       1
